# 05 Models — Logistic Regression — Women's

A linear model to add structural diversity to the ensemble. Tree models and neural nets are all nonlinear and highly correlated. Logistic regression makes fundamentally different errors, which benefits ensemble diversity.

**Inputs** (from S3 `04_preprocessing/womens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/logistic_regression/womens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

#### Functions

In [ ]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

#### Constants

In [ ]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "05_models/logistic_regression"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [ ]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [ ]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

## 2. Logistic Regression Setup

Linear models need feature scaling and NaN imputation. We use StandardScaler (fit per fold to avoid leakage) and fill NaN with 0 (natural default for difference features — means no difference between teams).

In [ ]:
LR_C = 1.0
LR_MAX_ITER = 1000

print(f"Logistic Regression parameters:")
print(f"  C (inverse regularization): {LR_C}")
print(f"  max_iter: {LR_MAX_ITER}")
print(f"  solver: lbfgs")
print(f"  NaN imputation: 0 (no difference)")
print(f"  Feature scaling: StandardScaler (per fold)")

## 3. LOGO Cross-Validation

In [ ]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols].fillna(0).values
    y_train = train.loc[train_mask, 'Label'].values
    X_val = train_original.loc[val_mask, feature_cols].fillna(0).values
    y_val = train_original.loc[val_mask, 'Label'].values
    
    if len(X_val) == 0:
        continue
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    
    model = LogisticRegression(C=LR_C, max_iter=LR_MAX_ITER, solver='lbfgs')
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_val)[:, 1]
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': 1
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

In [ ]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")

## 4. Train Final Model and Calibrate

In [ ]:
X_all = train[feature_cols].fillna(0).values
y_all = train['Label'].values

final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X_all)

final_model = LogisticRegression(C=LR_C, max_iter=LR_MAX_ITER, solver='lbfgs')
final_model.fit(X_all_scaled, y_all)

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

## 5. Generate Predictions

In [ ]:
X_s1 = final_scaler.transform(stage1[feature_cols].fillna(0).values)
stage1['Pred_logistic_regression'] = final_model.predict_proba(X_s1)[:, 1]
stage1['Pred_logistic_regression_calibrated'] = calibrator.predict(
    stage1['Pred_logistic_regression'].values
).clip(0.02, 0.98)

X_s2 = final_scaler.transform(stage2[feature_cols].fillna(0).values)
stage2['Pred_logistic_regression'] = final_model.predict_proba(X_s2)[:, 1]
stage2['Pred_logistic_regression_calibrated'] = calibrator.predict(
    stage2['Pred_logistic_regression'].values
).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_logistic_regression_calibrated'].min():.3f}, {stage1['Pred_logistic_regression_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_logistic_regression_calibrated'].min():.3f}, {stage2['Pred_logistic_regression_calibrated'].max():.3f}]")

stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_logistic_regression_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

## 6. Feature Coefficients

In [ ]:
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': final_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("Feature Coefficients (sorted by absolute value):")
for _, row in coef_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Coefficient']:+.4f}")

## 7. Save Outputs

In [ ]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_logistic_regression', 'Pred_logistic_regression_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_logistic_regression', 'Pred_logistic_regression_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

## 8. Summary

In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION MODEL SUMMARY — WOMEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
print(f"CV Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"\nTop 5 features (by |coefficient|):")
for _, row in coef_df.head().iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:+.4f}")